# Phase 06B.03A — Train temporal annotation audit

Accepts only human-authored train intervals; creates no intervals.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"src").is_dir()), None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from roadbuddy_common import save_json
from phase06a_common import sha256_file,sha256_json
from phase06b_common import validate_temporal_support_annotations
OUT=ROOT/"outputs/phase06b/temporal_annotation_audit"; OUT.mkdir(parents=True,exist_ok=True)
SUPPORT=ROOT/"data/phase06b/train_temporal_support.csv"; TRAIN=ROOT/"data/splits/phase01/train.csv"


In [ ]:
if not SUPPORT.is_file():
    save_json(OUT/"temporal_support_annotation.template.json",{"required_columns":["sample_id","group_id","video_sha256","support_start_sec","support_end_sec","annotation_source","annotator","adjudication_status","split"],"required_split":"train","note":"Human-authored intervals only"})
    save_json(OUT/"PHASE06B_03A_STATUS.json",{"status":"awaiting_temporal_annotation","missing":str(SUPPORT.relative_to(ROOT))})
    raise RuntimeError("Train-only temporal support annotations are missing")
train=pd.read_csv(TRAIN); group_map=dict(zip(train.sample_id.astype(str),train.group_id.astype(str)))
support=validate_temporal_support_annotations(pd.read_csv(SUPPORT),expected_train_ids=train.sample_id,expected_group_by_id=group_map)
report={"status":"pass","rows":len(support),"samples":support.sample_id.nunique(),"support_sha256":sha256_file(SUPPORT),"train_membership_sha256":sha256_json(sorted(train.sample_id.astype(str)))}
save_json(OUT/"temporal_annotation_audit.json",report); save_json(OUT/"temporal_annotation_manifest.json",report); save_json(OUT/"PHASE06B_03A_STATUS.json",{"status":"complete","audit_sha256":sha256_json(report)})
report
